Another proof of concept project to do the following objectives: 
1. Create and securely encrypt a confidential document using AES symmetric encryption 
2. Generate RSA public/private keys and use them to securely protect and recover the AES encryption key 
3. Decrypt and validate the original document while producing terminal-ready encryption workflow outputs.

In [2]:
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from tabulate import tabulate
from pathlib import Path
import hashlib

print ("\n=======================================")
print (" Enterprise Hybrid Encryption Workflow")
print ("\n=======================================")


 Enterprise Hybrid Encryption Workflow



In [ ]:
# ================================
# 1. Create working directory
# ================================

workspace = Path("secure_transfer_demo")
workspace.mkdir(exist_ok=True)

print(f"\nWorkspace directory created: {workspace.name}")


Workspace directory created: secure_transfer_demo


In [4]:
# ==========================================
# 2. Create confidential finance report
# ==========================================

report_path = workspace / "finance_report.txt"

report_content = """
CONFIDENTIAL FINANCE REPORT
-------------------------------
Target Company: Quantum Analytics
Estimated Acquisition Value: $24 Million
Legal Review Status: Pending
Internal Classification: Restricted
"""

report_path.write_text(report_content)
print(report_content)


CONFIDENTIAL FINANCE REPORT
-------------------------------
Target Company: Quantum Analytics
Estimated Acquisition Value: $24 Million
Legal Review Status: Pending
Internal Classification: Restricted



In [5]:
# =============================================
# 3. Generate AES symmetric encryption key
# =============================================

aes_key = Fernet.generate_key()
aes_cipher = Fernet(aes_key)

print("\n=== AES Session Key Generated ===")
print(aes_key.decode())


=== AES Session Key Generated ===
NmyBwKHiNUN5wmYc96qiCYHDWHXCKFmZBKgCnU0fJvQ=


In [6]:
# =============================================
# 4. Encrypt finance report using AES
# =============================================

original_data = report_path.read_bytes()

encrypted_report = aes_cipher.encrypt(original_data)

encrypted_report_path = workspace / "finance_report.encrypted"

encrypted_report_path.write_bytes(encrypted_report)

print("\nEncrypted finance report saved:")
print(encrypted_report_path)


Encrypted finance report saved:
secure_transfer_demo\finance_report.encrypted


In [7]:
# =============================================
# 5. Encrypt finance report using AES
# =============================================

private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

print("\n=== RSA Key Pair Generated ===")
print("RSA 2048-bit keys ready for secure key exchange")


=== RSA Key Pair Generated ===
RSA 2048-bit keys ready for secure key exchange


In [10]:
# =============================================
# 6. Encrypt AES key using RSA public key
# =============================================

encrypted_aes_key = public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

encrypted_key_path = workspace / "aes_key.secure"

encrypted_key_path.write_bytes(encrypted_aes_key)

print("\nProtected AES key saved:")
print(encrypted_key_path)


Protected AES key saved:
secure_transfer_demo\aes_key.secure


In [11]:
# =============================================
# 7. Recover AES key using RSA private key
# =============================================

recovered_aes_key = private_key.decrypt(
    encrypted_aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("\n=== AES Key Successfully Recovered ===")


=== AES Key Successfully Recovered ===


In [12]:
# =================================================
# 8. Attempt decryption with incorrect AES key
# =================================================

print("\nAttempting decryption with incorrect AES key...")

wrong_key = Fernet.generate_key()

wrong_cipher = Fernet(wrong_key)


try:
    wrong_cipher.decrypt(encrypted_report)

    print("Unexpected decryption success")

except Exception:

    print("Decryption Failed")
    print("Incorrect AES key cannot decrypt the report")


Attempting decryption with incorrect AES key...
Decryption Failed
Incorrect AES key cannot decrypt the report


In [18]:
# =================================================
# 9. Decrypt encrypted finance report correctly
# =================================================

print("\nApplying correct AES key...")

recovered_cipher = Fernet(recovered_aes_key)

decrypted_report = recovered_cipher.decrypt(
    encrypted_report
)

decrypted_report_path = workspace / "finance_report_recovered.txt"

decrypted_report_path.write_bytes(decrypted_report)

print("\nRecovered report saved:")
print(decrypted_report_path)

print("\n=== Successfully Recovered Report ===")
print(decrypted_report.decode())


Applying correct AES key...

Recovered report saved:
secure_transfer_demo\finance_report_recovered.txt

=== Successfully Recovered Report ===

CONFIDENTIAL FINANCE REPORT
-------------------------------
Target Company: Quantum Analytics
Estimated Acquisition Value: $24 Million
Legal Review Status: Pending
Internal Classification: Restricted



In [19]:
# ===================================
# 10. Validate report integrity
# ===================================

original_hash = hashlib.sha256(original_data).hexdigest()

recovered_hash = hashlib.sha256(decrypted_report).hexdigest()

validation = [
    ["Original Report Hash", original_hash],
    ["Recovered Report Hash", recovered_hash]
]

print("\n=== Integrity Validation ===")

print(tabulate(
    validation,
    headers=["Artifact", "SHA-256 Hash"],
    tablefmt="grid"
))


=== Integrity Validation ===
+-----------------------+------------------------------------------------------------------+
| Artifact              | SHA-256 Hash                                                     |
+=======================+==================================================================+
| Original Report Hash  | e7ba97b49f444a2e8833f470cf80882d34e1a7e4e69679a54d9ce25a1e5fb254 |
+-----------------------+------------------------------------------------------------------+
| Recovered Report Hash | e7ba97b49f444a2e8833f470cf80882d34e1a7e4e69679a54d9ce25a1e5fb254 |
+-----------------------+------------------------------------------------------------------+
